# Step 1: Install Required Dependencies
We install modern LangChain libraries, integration packages (including NVIDIA AI endpoints, Hugging Face integrations, and OpenAI/FAISS tools), and document parsers.

In [17]:
!pip -q install langchain langchain-community langchain-core langchain-nvidia-ai-endpoints langchain-text-splitters langchain-openai langchain-huggingface
!pip -q install pypdf
!pip -q install sentence_transformers
!pip install openai
!pip install tiktoken

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 125.8/125.8 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 571.7/571.7 kB 17.1 MB/s eta 0:00:00


# Step 2: Ensure Latest LangChain & NVIDIA Endpoint Package Installation

In [4]:
%pip install -qU langchain-nvidia-ai-endpoints langchain

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.5/64.5 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 161.5/161.5 kB 7.8 MB/s eta 0:00:00


# Step 3: Install Utility Packages
Installing `faiss-cpu` for vector indexing, `tokenizers` for text encoding, and `unstructured` for processing raw text data.

In [2]:
!pip install tokenizers
!pip install faiss-cpu
!pip -q install unstructured

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 96.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.5/981.5 kB 18.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.5/66.5 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 50.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 109.9/109.9 kB 11.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 608.4/608.4 kB 45.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 453.8/453.8 kB 40.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 155.6/155.6 kB 15.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 167.9/167.9 kB 16.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 105.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 169.6/169.6 kB 17.7 MB/s eta 0:00:00


# Step 4: Install Essential Mathematical and NLP Packages

In [3]:
!pip install numpy
!pip install nltk

# Step 5: Import Required Libraries
Importing standard utilities, document loaders, chunk splitters, vector stores, embedding classes, NVIDIA endpoint clients, and classic Retrieval chains.

In [19]:
import sys
import os
import torch
import textwrap
from langchain_community.document_loaders import UnstructuredURLLoader
from langchain_text_splitters import CharacterTextSplitter
from langchain_openai import OpenAIEmbeddings
from langchain_nvidia import ChatNVIDIA
from langchain_community.vectorstores import FAISS
from langchain_classic.chains import RetrievalQAWithSourcesChain
from langchain_huggingface import HuggingFaceEmbeddings

# Step 6: Download NLTK Tokenization Models

In [20]:
import nltk
nltk.download('punkt')
nltk.download('averaged_perceptron_tagger')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /root/nltk_data...
[nltk_data]   Unzipping taggers/averaged_perceptron_tagger.zip.


True

# Step 7: Configure Environment Credentials
Setting up the API Key for the NVIDIA AI Foundation endpoints.

In [21]:
os.environ['NVIDIA_API_KEY'] = "API_KEY"

# Step 8: Define Knowledge Source URLs
Providing the target URLs containing the blogs and technical articles to ingest.

In [23]:
URLs =[
    "https://blog.gopenai.com/paper-review-llama-2-open-foundation-and-fine-tuned-chat-models-23e539522acb",
    "https://www.mosaicml.com/blog/mpt-7b",
    "https://stability.ai/blog/stability-ai-launches-the-first-of-its-stablelm-suite-of-language-models",
    "https://lmsys.org/blog/2023-03-30-vicuna/"
]

# Step 9: Load Document Data from URLs

In [24]:
loader = UnstructuredURLLoader(urls=URLs)
data = loader.load()

# Step 10: Verify Loaded Document Count

In [25]:
len(data)

4

# Step 11: Configure Text Splitter Params
Instantiating a character text splitter with overlapping token spaces to prevent contextual loss at boundaries.

In [26]:
text_splitter = CharacterTextSplitter(
    separator = "\n",
    chunk_size = 1000,
    chunk_overlap  = 200
)

# Step 12: Split Documents into Smaller Chunks

In [27]:
text_chunks = text_splitter.split_documents(data)

# Step 13: Count Resulting Document Chunks

In [28]:
len(text_chunks)

61

# Step 14: Inspect a Sample Chunk

In [29]:
text_chunks[0]

Document(metadata={'source': 'https://blog.gopenai.com/paper-review-llama-2-open-foundation-and-fine-tuned-chat-models-23e539522acb'}, page_content='Please enable cookies.\nSorry, you have been blocked\nYou are unable to access medium.com\nWhy have I been blocked?\nThis website is using a security service to protect itself from online attacks. The action you just performed triggered the security solution. There are several actions that could trigger this block including submitting a certain word or phrase, a SQL command or malformed data.\nWhat can I do to resolve this?\nYou can email the site owner to let them know you were blocked. Please include what you were doing when this page came up and the Cloudflare Ray ID found at the bottom of this page.\nCloudflare Ray ID: a3977be09d39c158 • Your IP: 34.74.175.134 • Performance & security by Cloudflare')

# Step 15: Initialize Embedding Model
Setting up the HuggingFace `all-MiniLM-L6-v2` local model to map chunks to high-dimensional space.

In [30]:
embedding_model = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

# Step 16: Verify Embedding Dimension Response

In [31]:
query_result = embedding_model.embed_query("Hello World")
len(query_result)

384

# Step 17: Inspect Sample Embedding Vectors

In [33]:
query_result[:5]

[-0.03447728976607323,
 0.03102319873869419,
 0.0067349751479923725,
 0.026109017431735992,
 -0.03936200216412544]

# Step 18: Build and Index the FAISS Vector Database
Passing chunked texts and the HuggingFace embedding instance to generate our vector indices locally.

In [34]:
vectorstores = FAISS.from_documents(text_chunks, embedding_model)

# Step 19: Initialize ChatNVIDIA LLM Endpoint
Instantiating the `openai/gpt-oss-20b` remote model using client-side configurations (timeouts and tokens).

In [43]:
llm = ChatNVIDIA(
    model = 'openai/gpt-oss-20b',
    api_key = os.environ['NVIDIA_API_KEY'],
    temperature = 0.1,
    max_tokens=1024,
    top_p=1,
    timeout=200
)

/tmp/ipykernel_623/1109644556.py:1: DeprecationWarning: The 'max_tokens' parameter is deprecated and will be removed in a future version. Please use 'max_completion_tokens' instead.
  llm = ChatNVIDIA(


# Step 20: Test Basic Chat LLM Output

In [36]:
llm.invoke("Please provide a concise summary of the article")


AIMessage(content='I’m happy to help! Could you please share the article (or a link to it) so I can read it and provide a concise summary?', additional_kwargs={'reasoning_content': 'We need to summarize an article. But the article is not provided. The user says "Please provide a concise summary of the article". We need to ask for the article or clarify. Probably we should ask for the article text or link.', 'reasoning': 'We need to summarize an article. But the article is not provided. The user says "Please provide a concise summary of the article". We need to ask for the article or clarify. Probably we should ask for the article text or link.', '_reasoning_api_fields': ['reasoning_content', 'reasoning']}, response_metadata={'role': 'assistant', 'content': 'I’m happy to help! Could you please share the article (or a link to it) so I can read it and provide a concise summary?', 'refusal': None, 'annotations': None, 'audio': None, 'function_call': None, 'tool_calls': [], 'reasoning': 'We

# Step 21: Setup RAG Pipeline and Run Query
Creating a classic RetrievalQA Chain with source references, querying it about Vicuna, and retrieving responses.

In [44]:
chain = RetrievalQAWithSourcesChain.from_llm(llm=llm, retriever=vectorstores.as_retriever())

# Using modern .invoke method to avoid deprecation warnings
result = chain.invoke({"question" : "How good is Vicuna"}, return_only_outputs=True)

# Step 22: Display Sample Q&A Source Result

In [45]:
result['answer']

'**Answer**\n\nVicuna‑13B is a high‑quality open‑source chatbot that, according to a series of evaluations using GPT‑4 as a judge, performs at a level comparable to commercial models such as ChatGPT and Google Bard.  \n\n* **Overall quality** – Vicuna‑13B achieves more than **90\u202f% of the quality** of ChatGPT and Bard, and it outperforms other open‑source baselines (LLaMA, Alpaca) in over **90\u202f% of the test cases**.  \n* **Preference by GPT‑4** – In a head‑to‑head comparison, GPT‑4 prefers Vicuna over all other open‑source models in **>90\u202f% of the questions**.  \n* **Score relative to ChatGPT** – When GPT‑4 assigns a numeric score (0‑10) to each answer, Vicuna’s total score is **92\u202f% of ChatGPT’s** (Table\u202f2 in the blog).  \n* **Strengths** – Vicuna produces detailed, well‑structured responses that are on par with ChatGPT for many conversational tasks.  \n* **Limitations** – The model still struggles with reasoning, mathematics, and factual accuracy, and it has n

# Step 23: Perform Secondary Model Query
Querying the active pipeline for detailed technical features of the MPT-7b model.

In [46]:
result=chain({"question": "Can you please share some details about MPT-7b Model"}, return_only_outputs=True)
result['answer']

'**MPT‑7B – Mosaic Pretrained Transformer (7\u202fB parameters)**  \n\n- **Architecture** – A GPT‑style decoder‑only transformer that uses performance‑optimised layer implementations, architectural changes for greater training stability, and ALiBi positional embeddings to remove context‑length limits.  \n- **Training** – Built on the MosaicML platform with zero human intervention, trained for 9.5\u202fdays on 440 GPUs. The platform automatically detected and recovered from four hardware failures during training.  \n- **Data** – Trained on ~1\u202ftrillion tokens (≈1\u202fT), far more than many comparable open‑source models.  \n- **Performance** – Achieves 40–60\u202f% MFU (model‑flop utilisation) while maintaining stable loss curves.  \n- **Deployment** – Can be served via standard HuggingFace pipelines or FasterTransformer, and is licensed for commercial use.  \n- **Variants** – MosaicML also released finetuned versions: MPT‑7B‑Instruct, MPT‑7B‑Chat, and MPT‑7B‑StoryWriter‑65k+ (suppo

# Step 24: Interactive CLI Prompt Chat loop
Runs a simple input console inside your environment for real-time querying.

In [49]:
import sys

while True:
    query = input("Prompt: ")
    if query == 'exit':
        print('Exiting')
        break
    if query == '':
        continue
    result = chain.invoke({'question': query})
    print("Answer: " + result["answer"])
    print("-" * 50)

Prompt: exit
Exiting


## Summary & Conclusion

### Overall System Workflow
This notebook sets up a fully functioning local **Retrieval-Augmented Generation (RAG)** pipeline. Here is the sequential process flow:
1. **Data Ingestion:** Documents are pulled dynamically from technical URLs.
2. **Document Splitting:** The system breaks long articles down into overlapping chunks of 1000 characters to retain semantic structure.
3. **Embedding Generation:** The text chunks are vectorized using a HuggingFace local embedding model (`all-MiniLM-L6-v2`).
4. **Indexing (FAISS):** The vector dimensions are organized inside a fast-retrieval FAISS search space index.
5. **Retrieval & Inference:** When a user enters a query, relevant text chunks are extracted from FAISS, fed into the context template of the NVIDIA `openai/gpt-oss-20b` LLM endpoint, and returned in natural language along with citations.

### System Architecture
```
                               +------------------+
                               | Target Web URLs  |
                               +--------+---------+
                                        |
                                        | (Step 1: Document Loading)
                                        v
                               +------------------+
                               | Document Parsers |
                               +--------+---------+
                                        |
                                        | (Step 2: Split into chunks)
                                        v
                               +------------------+
                               |   Text Chunks    |
                               +--------+---------+
                                        |
                                        | (Step 3: Embed text via HuggingFace)
                                        v
                               +------------------+
                               | Embedding Model  |
                               +--------+---------+
                                        |
                                        v
                               +------------------+
                               | FAISS Vector DB  |
                               +--------+---------+
                                        ^
  User Query                            | (Step 4: Retrieve relative context)
  [ "How good is Vicuna?" ] ------------+
         |                              |
         v                              |
  +------+------------------------------+---------+
  | Formulated LLM Context Template               |
  | (Context + original prompt questions)         |
  +------+----------------------------------------+
         |
         | (Step 5: Send context package to GPU Endpoints)
         v
  +------+------------------+
  |   NVIDIA Chat LLM       |
  | (openai/gpt-oss-20b)     |
  +------+------------------+
         |
         | (Step 6: Return final synthesis with citations)
         v
  +------+------------------+
  | Generated User Output   |
  +-------------------------+
```